<a href="https://colab.research.google.com/github/BautistaLegno/space-up-analytics/blob/main/Space%20Up%20Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipyleaflet import Map, Marker, basemaps, Rectangle
from geopy.geocoders import Nominatim
from google.colab import drive
from google.colab import files
import datetime
import requests

# ====================================================================
# 1. INICIALIZACIÓN
# ====================================================================
print("Inicializando Motor Analítico de Space Up...")
np.random.seed(42)
fechas_mes = pd.date_range(start="2020-01-01", end="2028-12-01", freq="MS")
df_mes = pd.DataFrame({'fecha': fechas_mes, 'humedad_superficial': 0.20, 'temp_suelo': 18.0, 'temp_aire': 22.0, 'precipitacion': 0.0})
df = df_mes.set_index('fecha').resample('D').interpolate(method='cubic').reset_index()
df_real_base = df.copy()

reporte_texto = ""
HOY = pd.to_datetime("2026-11-06")

# ====================================================================
# 2. MAPA INTERACTIVO Y BÚSQUEDA
# ====================================================================
mapa = Map(center=[-33.0, -61.5], zoom=6, basemap=basemaps.Esri.WorldImagery, layout=widgets.Layout(height='380px', margin='10px 0', border='1px solid #ccc', border_radius='5px'))
limites_zona_nucleo = ((-35.0, -64.0), (-31.0, -59.0))
recuadro = Rectangle(bounds=limites_zona_nucleo, color='#ff7800', fill_color='#ff7800', fill_opacity=0.2, weight=2)
mapa.add_layer(recuadro)

marcador = Marker(location=[0, 0], draggable=True)
salida_busqueda = widgets.Output()

#SELECCION MANUAL
# Geocodificador
geolocador = Nominatim(user_agent="space_up_agro")

#Historial de Ubicaciones
historial_opciones = [('Historial de búsquedas...', None)]

dropdown_historial = widgets.Dropdown(
    options=historial_opciones,
    description='Historial:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%', margin='5px 0 0 0')
)

def agregar_al_historial(nombre, lat, lon):
    etiqueta = f"{nombre} ({lat:.4f}, {lon:.4f})"
    coords = (lat, lon)
    for _, val in historial_opciones:
        if val == coords:
            return
    historial_opciones.append((etiqueta, coords))
    dropdown_historial.options = list(historial_opciones)

def seleccionar_del_historial(change):
    if change['type'] == 'change' and change['name'] == 'value':
        coords = change['new']
        if coords is not None:
            marcador.location = coords
            mapa.center = coords
            mapa.zoom = 12
            if marcador not in mapa.layers:
                mapa.add_layer(marcador)
            with salida_busqueda:
                clear_output()
                display(widgets.HTML(f"<div style='color: #052c65; background-color: #cfe2ff; padding: 8px; border-radius: 5px; margin-top: 5px;'>📍 <b>Seleccionado del Historial</b> | Lat: {coords[0]:.4f}, Lon: {coords[1]:.4f}</div>"))

dropdown_historial.observe(seleccionar_del_historial)

#Interacción con Mapa (Clic)
def actualizar_marcador(**kwargs):
    if kwargs.get('type') == 'click':
        coords = kwargs.get('coordinates')
        marcador.location = coords
        if marcador not in mapa.layers:
            mapa.add_layer(marcador)

        # Geocodificación inversa para obtener el nombre del lugar
        try:
            ubicacion = geolocador.reverse((coords[0], coords[1]), timeout=3)
            nombre_lugar = ubicacion.address.split(',')[0] if ubicacion else "Lote Seleccionado"
        except:
            nombre_lugar = "Lote Seleccionado"

        agregar_al_historial(nombre_lugar, coords[0], coords[1])

        with salida_busqueda:
            clear_output()
            display(widgets.HTML(f"<div style='color: #052c65; background-color: #cfe2ff; padding: 8px; border-radius: 5px; margin-top: 5px;'>📍 <b>{nombre_lugar}</b> | Lat: {coords[0]:.4f}, Lon: {coords[1]:.4f}</div>"))

mapa.on_interaction(actualizar_marcador)

#Buscador por Texto
buscador_texto = widgets.Text(placeholder='Ej: Cruz Alta', description='Buscar:', style={'description_width': 'initial'}, layout=widgets.Layout(width='70%'))
boton_buscar = widgets.Button(description='Ir', button_style='info', icon='search', layout=widgets.Layout(width='25%'))

def buscar_ubicacion(b):
    with salida_busqueda:
        clear_output()
        texto = buscador_texto.value.strip().replace(" ", "")
        ubicacion = geolocador.geocode({'postalcode': texto, 'country': 'Argentina'}) if texto.isdigit() else geolocador.geocode(buscador_texto.value.strip(), country_codes='ar')
        if ubicacion:
            marcador.location = (ubicacion.latitude, ubicacion.longitude)
            mapa.center = (ubicacion.latitude, ubicacion.longitude)
            mapa.zoom = 12
            if marcador not in mapa.layers:
                mapa.add_layer(marcador)

            partes_direccion = ubicacion.address.split(',')
            nombre_lugar = partes_direccion[0].strip()

            if nombre_lugar.isdigit() and len(partes_direccion) > 1:
                nombre_lugar = partes_direccion[1].strip()

            agregar_al_historial(nombre_lugar, ubicacion.latitude, ubicacion.longitude)

            display(widgets.HTML(f"<div style='color: #0f5132; background-color: #d1e7dd; padding: 8px; border-radius: 5px; margin-top: 5px;'>📍 <b>{nombre_lugar}</b> | Lat: {ubicacion.latitude:.4f}, Lon: {ubicacion.longitude:.4f}</div>"))
        else:
            display(widgets.HTML("<div style='color: #842029; background-color: #f8d7da; padding: 8px; border-radius: 5px; margin-top: 5px;'>⚠️ No encontrado.</div>"))

boton_buscar.on_click(buscar_ubicacion)

# Contenedor Vertical de Búsqueda + Historial
caja_busqueda = widgets.VBox([
    widgets.HBox([buscador_texto, boton_buscar], layout=widgets.Layout(justify_content='space-between')),
    dropdown_historial
])

# Consulta Puntual por Fecha
consulta_fecha = widgets.DatePicker(description="Día exacto:", style={'description_width': 'initial'}, layout=widgets.Layout(width='60%'))
boton_consultar_fecha = widgets.Button(description="Ver Datos Día", button_style='info', icon='search', layout=widgets.Layout(width='35%'))
salida_consulta_fecha = widgets.Output()

def buscar_humedad_puntual(b):
    global df
    with salida_consulta_fecha:
        clear_output()
        if consulta_fecha.value:
            fecha_buscada = pd.to_datetime(consulta_fecha.value)
            resultado = df[df['fecha'].dt.date == fecha_buscada.date()]

            if not resultado.empty:
                hum = resultado['humedad_superficial'].values[0]
                temp_s = resultado['temp_suelo'].values[0]
                lluvia = resultado['precipitacion'].values[0]

                if fecha_buscada <= HOY:
                    tipo_dato, color_f, color_t = "Datos Reales", "#d1e7dd", "#0f5132"
                else:
                    tipo_dato, color_f, color_t = "Proyección IA", "#fff3cd", "#664d03"

                display(widgets.HTML(f"<div style='color: {color_t}; background-color: {color_f}; padding: 8px; border-radius: 5px; margin-top: 5px; font-size: 14px;'>📅 <b>{fecha_buscada.strftime('%d/%m/%Y')}</b> <i>({tipo_dato})</i><br>💧 Humedad: <b>{hum:.3f} cm³/cm³</b> | 🌡️ Temp. Suelo: <b>{temp_s:.1f} °C</b><br>🌧️ Lluvia: <b>{lluvia:.1f} mm</b></div>"))
            else:
                display(widgets.HTML("<div style='color: #842029; background-color: #f8d7da; padding: 8px; border-radius: 5px; margin-top: 5px;'>⚠️ Fecha fuera de rango. Ejecutá un Análisis de Siembra primero.</div>"))
        else:
            display(widgets.HTML("<div style='color: #842029; background-color: #f8d7da; padding: 8px; border-radius: 5px; margin-top: 5px;'>⚠️ Ingresá una fecha.</div>"))

boton_consultar_fecha.on_click(buscar_humedad_puntual)
caja_consulta_puntual = widgets.HBox([consulta_fecha, boton_consultar_fecha], layout=widgets.Layout(justify_content='space-between', margin='10px 0 0 0'))

# ====================================================================
# 3. CONTROLES SECCIÓN 1 (SIEMBRA ACTUAL)
# ====================================================================
selector_cultivo = widgets.Dropdown(
    options=[('Trigo (Jun - Dic)', ('-06-15', '-12-10')), ('Cebada (Jun - Dic)', ('-06-01', '-12-05')), ('Maíz Temprano (Sep - Abr)', ('-09-15', '-04-15')), ('Girasol (Oct - Mar)', ('-10-15', '-03-15')), ('Soja de 1ra (Nov - May)', ('-11-01', '-05-15')), ('Soja de 2da (Dic - May)', ('-12-15', '-05-20')), ('Maíz Tardío (Dic - Jul)', ('-12-10', '-07-15')), ('Sorgo (Nov - Abr)', ('-11-15', '-04-30')), ('Pastura / Cultivo de Servicio (Mar - Sep)', ('-03-15', '-09-15'))],
    description='Cultivo:', style={'description_width': 'initial'}, layout=widgets.Layout(width='100%', margin='0 0 10px 0')
)
selector_fecha_siembra = widgets.DatePicker(description='Fecha Siembra:', style={'description_width': 'initial'}, layout=widgets.Layout(width='100%', margin='0 0 10px 0'))

def actualizar_fecha_default(*args):
    try: selector_fecha_siembra.value = pd.to_datetime(f"{HOY.year}{selector_cultivo.value[0]}").date()
    except: pass
selector_cultivo.observe(actualizar_fecha_default, 'value')
actualizar_fecha_default()

entrada_napa = widgets.BoundedFloatText(value=2.5, min=0.0, max=10.0, step=0.1, description='Napa (m):', style={'description_width': 'initial'}, layout=widgets.Layout(width='100%', margin='0 0 10px 0'))
slider_umbral = widgets.FloatSlider(value=0.20, min=0.10, max=0.30, step=0.01, description='Buffer (cm³/cm³):', style={'description_width': 'initial'}, layout=widgets.Layout(width='100%', margin='0 0 15px 0'))
boton_analizar_siembra = widgets.Button(description='Descargar API y Analizar Siembra', button_style='success', icon='cloud-download', layout=widgets.Layout(width='100%', height='45px', margin='0 0 10px 0'))

# ====================================================================
# 4. CONTROLES SECCIÓN 2 (HISTÓRICO Y ROTACIÓN)
# ====================================================================
selector_ano_hist = widgets.Dropdown(
    options=[('Campaña 2021', 2021), ('Campaña 2022 (Sequía)', 2022), ('Campaña 2023', 2023), ('Campaña 2024', 2024), ('Campaña 2025', 2025)],
    value=2024,
    description='Campaña:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%', margin='0 0 10px 0')
)
fecha_siembra_hist = widgets.DatePicker(description='Siembra Real:', style={'description_width': 'initial'}, layout=widgets.Layout(width='100%', margin='0 0 8px 0'))
fecha_cosecha_hist = widgets.DatePicker(description='Cosecha Real:', style={'description_width': 'initial'}, layout=widgets.Layout(width='100%', margin='0 0 8px 0'))

opciones_rotacion = ['Soja', 'Maíz', 'Trigo', 'Sorgo', 'Girasol', 'Pastura / Cultivo de Servicio']
rot_1 = widgets.Dropdown(options=opciones_rotacion, value='Soja', description=f'Año {HOY.year - 5}:', style={'description_width': 'initial'}, layout=widgets.Layout(width='100%', margin='0 0 8px 0'))
rot_2 = widgets.Dropdown(options=opciones_rotacion, value='Soja', description=f'Año {HOY.year - 4}:', style={'description_width': 'initial'}, layout=widgets.Layout(width='100%', margin='0 0 8px 0'))
rot_3 = widgets.Dropdown(options=opciones_rotacion, value='Maíz', description=f'Año {HOY.year - 3}:', style={'description_width': 'initial'}, layout=widgets.Layout(width='100%', margin='0 0 8px 0'))
rot_4 = widgets.Dropdown(options=opciones_rotacion, value='Soja', description=f'Año {HOY.year - 2}:', style={'description_width': 'initial'}, layout=widgets.Layout(width='100%', margin='0 0 8px 0'))
rot_5 = widgets.Dropdown(options=opciones_rotacion, value='Trigo', description=f'Año {HOY.year - 1}:', style={'description_width': 'initial'}, layout=widgets.Layout(width='100%', margin='0 0 8px 0'))

# NUEVO: DICCIONARIO DE FECHAS TEÓRICAS SEGÚN CULTIVO HISTÓRICO
fechas_teoricas_hist = {
    'Soja': ('-11-01', '-05-15'),
    'Maíz': ('-09-15', '-04-15'),
    'Trigo': ('-06-15', '-12-10'),
    'Sorgo': ('-11-15', '-04-30'),
    'Girasol': ('-10-15', '-03-15'),
    'Pastura / Cultivo de Servicio': ('-03-15', '-09-15')
}

def sincronizar_fechas_hist(*args):
    try:
        y = selector_ano_hist.value
        # Detectar qué cultivo corresponde al año seleccionado
        if y == HOY.year - 5: cultivo_hist = rot_1.value
        elif y == HOY.year - 4: cultivo_hist = rot_2.value
        elif y == HOY.year - 3: cultivo_hist = rot_3.value
        elif y == HOY.year - 2: cultivo_hist = rot_4.value
        elif y == HOY.year - 1: cultivo_hist = rot_5.value
        else: cultivo_hist = 'Soja'

        mes_inicio_str, mes_fin_str = fechas_teoricas_hist[cultivo_hist]

        m_inicio = int(mes_inicio_str.split('-')[1])
        m_fin = int(mes_fin_str.split('-')[1])

        # Cambio de año (ejemplo siembra en Nov, cosecha en May)
        y_fin = y + 1 if m_inicio > m_fin else y

        fecha_siembra_hist.value = pd.to_datetime(f"{y}{mes_inicio_str}").date()
        fecha_cosecha_hist.value = pd.to_datetime(f"{y_fin}{mes_fin_str}").date()
    except:
        pass

# Conectar el sincronizador a los cambios de año y de cada menú de rotación
selector_ano_hist.observe(sincronizar_fechas_hist, 'value')
rot_1.observe(sincronizar_fechas_hist, 'value')
rot_2.observe(sincronizar_fechas_hist, 'value')
rot_3.observe(sincronizar_fechas_hist, 'value')
rot_4.observe(sincronizar_fechas_hist, 'value')
rot_5.observe(sincronizar_fechas_hist, 'value')
sincronizar_fechas_hist() # Ejecución inicial

boton_analizar_historico = widgets.Button(description='Evaluar Historial', button_style='info', icon='history', layout=widgets.Layout(width='100%', height='45px', margin='0 0 10px 0'))
boton_exportar = widgets.Button(description='Descargar Reporte', button_style='primary', icon='download', layout=widgets.Layout(width='100%', height='45px'))

out_siembra = widgets.Output(layout=widgets.Layout(margin='20px 0 0 0'))
out_historico = widgets.Output(layout=widgets.Layout(margin='20px 0 0 0'))

req_cultivo = {
    'Trigo': {'temp': 10, 'hum': 0.18, 'vent': [5, 6, 7, 8]},
    'Cebada': {'temp': 10, 'hum': 0.18, 'vent': [5, 6, 7, 8]},
    'Maíz Temprano': {'temp': 12, 'hum': 0.20, 'vent': [8, 9, 10]},
    'Girasol': {'temp': 12, 'hum': 0.18, 'vent': [9, 10, 11]},
    'Soja de 1ra': {'temp': 15, 'hum': 0.22, 'vent': [10, 11, 12]},
    'Soja de 2da': {'temp': 18, 'hum': 0.20, 'vent': [12, 1]},
    'Maíz Tardío': {'temp': 17, 'hum': 0.22, 'vent': [11, 12, 1, 2]},
    'Sorgo': {'temp': 18, 'hum': 0.20, 'vent': [10, 11, 12]},
    'Pastura': {'temp': 10, 'hum': 0.15, 'vent': [2, 3, 4, 8, 9, 10]}
}

# ====================================================================
# 5. FUNCIONES DE CONEXIÓN Y DATOS
# ====================================================================
def descargar_api_con_respaldo(lat, lon, widget_salida):
    global df_real_base
    HOY = pd.to_datetime("today").normalize()

    # NASA POWER suele tener un retraso de 2 a 4 días en sus registros de reanálisis
    fecha_fin_api = HOY - pd.Timedelta(days=5)
    start_date = "20200101"
    end_date = fecha_fin_api.strftime('%Y%m%d')

    # URL del endpoint de NASA POWER (sección AG = Agrometeorology)
    url_api = (
        f"https://power.larc.nasa.gov/api/temporal/daily/point?"
        f"parameters=GWETTOP,TS,T2M,PRECTOTCORR&"
        f"community=AG&"
        f"longitude={lon}&latitude={lat}&"
        f"start={start_date}&end={end_date}&"
        f"format=JSON"
    )

    try:
        respuesta = requests.get(url_api, timeout=15)
        respuesta.raise_for_status()
        datos = respuesta.json()

        # Extraer las series temporales del diccionario 'properties' -> 'parameter'
        parametros = datos['properties']['parameter']
        fechas_raw = list(parametros['T2M'].keys())

        # Crear el DataFrame inicial
        df_real = pd.DataFrame({
            'fecha': pd.to_datetime(fechas_raw, format='%Y%m%d'),
            'humedad_superficial': list(parametros['GWETTOP'].values()),
            'temp_suelo': list(parametros['TS'].values()),
            'temp_aire': list(parametros['T2M'].values()),
            'precipitacion': list(parametros['PRECTOTCORR'].values())
        })

        POROSIDAD_PROMEDIO = 0.42  # Ajustable según el suelo
        df_real['humedad_superficial'] = df_real['humedad_superficial'] * POROSIDAD_PROMEDIO

        # 1. Limpieza de valores nulos nativos de la NASA (-999.0)
        columnas_numericas = ['humedad_superficial', 'temp_suelo', 'temp_aire', 'precipitacion']
        for col in columnas_numericas:
            df_real[col] = df_real[col].replace(-999.0, np.nan)
            df_real[col] = pd.to_numeric(df_real[col], errors='coerce')

        # 2. Reajuste de desfase de fechas al día de hoy
        delta = HOY - df_real['fecha'].max()
        df_real['fecha'] = df_real['fecha'] + delta

        # 3. Interpolación de datos faltantes e imputación
        df_real['humedad_superficial'] = df_real['humedad_superficial'].interpolate().fillna(0.20)
        df_real['temp_suelo'] = df_real['temp_suelo'].interpolate().fillna(15.0)
        df_real['temp_aire'] = df_real['temp_aire'].interpolate().fillna(18.0)
        df_real['precipitacion'] = df_real['precipitacion'].fillna(0.0)

        df_real_base = df_real[['fecha', 'humedad_superficial', 'temp_suelo', 'temp_aire', 'precipitacion']].copy()

        # Guardar archivo local
        nombre_archivo = f"datos_meteo_NASA_lat_{lat:.2f}_lon_{lon:.2f}.csv"
        df_real_base.to_csv(nombre_archivo, index=False, encoding="utf-8-sig")

    except Exception as e:
        with widget_salida:
            display(widgets.HTML("<div style='color: #842029; background-color: #f8d7da; padding: 15px; border-radius: 8px;'>⚠️ <b>Aviso de Red:</b> API inactiva. Utilizando Simulador Climático de Respaldo Estacional.</div>"))

        np.random.seed(888)
        fechas_backup = pd.date_range(start="2020-01-01", end=HOY, freq="D")
        lluvias = np.random.choice([0]*17 + [8, 15, 30], size=len(fechas_backup))
        meses = fechas_backup.month
        temp_s_back = 18 + 8 * np.sin(np.pi * (meses - 10) / 6) + np.random.normal(0, 2, len(fechas_backup))
        temp_a_back = 22 + 10 * np.sin(np.pi * (meses - 10) / 6) + np.random.normal(0, 2, len(fechas_backup))

        df_real = pd.DataFrame({
            'fecha': fechas_backup,
            'humedad_superficial': np.random.uniform(0.12, 0.35, len(fechas_backup)),
            'temp_suelo': temp_s_back,
            'temp_aire': temp_a_back,
            'precipitacion': lluvias
        })
        df_real['humedad_superficial'] = df_real['humedad_superficial'].rolling(window=10, min_periods=1).mean()
        df_real_base = df_real.copy()
        np.random.seed(None)

# ====================================================================
# 6. FUNCIONES DE ANÁLISIS VISUAL
# ====================================================================
def procesar_siembra(b):
    global df
    with out_siembra:
        clear_output(wait=True)
        if marcador not in mapa.layers:
            display(widgets.HTML("<div style='color: #842029; background-color: #f8d7da; padding: 15px; border-radius: 8px;'>⚠️ <b>ALTO:</b> Marcá el lote en el mapa primero.</div>"))
            return

        display(widgets.HTML("<div style='color: #052c65; background-color: #cfe2ff; padding: 15px; border-radius: 8px;'>📡 <i>Conectando con servidores meteorológicos y analizando ventana...</i></div>"))
        lat, lon = marcador.location
        inicio = pd.to_datetime(selector_fecha_siembra.value) if selector_fecha_siembra.value else pd.to_datetime(HOY)

        descargar_api_con_respaldo(lat, lon, out_siembra)

        cultivo_key = next((k for k in req_cultivo.keys() if k in selector_cultivo.label), 'Soja de 1ra')
        t_min, h_min = req_cultivo[cultivo_key]['temp'], req_cultivo[cultivo_key]['hum']
        meses_permitidos = req_cultivo[cultivo_key]['vent']
        mes_siembra = inicio.month

        en_fecha = mes_siembra in meses_permitidos

        mes_inicio_str, mes_fin_str = selector_cultivo.value
        m_inicio_ciclo = int(mes_inicio_str.split('-')[1])
        m_fin_ciclo = int(mes_fin_str.split('-')[1])

        ano_base = HOY.year
        ano_fin_teorico = ano_base + 1 if m_inicio_ciclo > m_fin_ciclo else ano_base
        duracion_ciclo = pd.to_datetime(f"{ano_fin_teorico}{mes_fin_str}") - pd.to_datetime(f"{ano_base}{mes_inicio_str}")
        fin = inicio + duracion_ciclo

        if fin > HOY:
            fechas_futuras = pd.date_range(start=df_real_base['fecha'].max() + pd.Timedelta(days=1), end=fin + pd.Timedelta(days=45), freq="D")
            dias_futuros = len(fechas_futuras)

            historico_lluvias = df_real_base['precipitacion'].values
            estados = (historico_lluvias >= 1.0).astype(int)

            transiciones_0 = np.sum(estados[:-1] == 0)
            transiciones_1 = np.sum(estados[:-1] == 1)
            p_01 = np.sum((estados[:-1] == 0) & (estados[1:] == 1)) / transiciones_0 if transiciones_0 > 0 else 0.15
            p_11 = np.sum((estados[:-1] == 1) & (estados[1:] == 1)) / transiciones_1 if transiciones_1 > 0 else 0.30

            cantidades_lluvia = historico_lluvias[historico_lluvias >= 1.0]
            if len(cantidades_lluvia) == 0: cantidades_lluvia = np.array([5, 10, 15, 20])

            lluvias_futuras = []
            estado_actual = 0 if df_real_base['precipitacion'].iloc[-1] < 1.0 else 1

            np.random.seed(999)
            for _ in range(dias_futuros):
                if estado_actual == 0:
                    estado_actual = np.random.choice([0, 1], p=[1 - p_01, p_01])
                else:
                    estado_actual = np.random.choice([0, 1], p=[1 - p_11, p_11])

                if estado_actual == 1:
                    lluvias_futuras.append(np.random.choice(cantidades_lluvia))
                else:
                    lluvias_futuras.append(0.0)

            lluvias_futuras = np.array(lluvias_futuras)

            hum_futura = []
            temp_s_futura = []
            temp_a_futura = []

            h_actual = df_real_base['humedad_superficial'].iloc[-1]
            t_s_actual = df_real_base['temp_suelo'].iloc[-1]

            for i, lluvia in enumerate(lluvias_futuras):
                mes_futuro = fechas_futuras[i].month

                if lluvia > 0:
                    h_actual = min(0.42, h_actual + (lluvia * 0.012))
                else:
                        # Evaporación estacional adaptada a ET0 (Secado rápido en verano / lento en invierno)
                    if mes_futuro in [11, 12, 1, 2]:
                      tasa_secado = 0.85  # Verano (alta evapotranspiración)
                    elif mes_futuro in [5, 6, 7, 8]:
                      tasa_secado = 0.97  # Invierno (baja evapotranspiración)
                    else:
                      tasa_secado = 0.91  # Primavera / Otoño

                    h_actual = max(0.05, h_actual * tasa_secado)

                hum_futura.append(h_actual)


                t_base_s = 18 + 8 * np.sin(np.pi * (mes_futuro - 10) / 6)

                if lluvia > 0:
                    t_objetivo = t_base_s - np.random.uniform(2.0, 4.0)
                else:
                    t_objetivo = t_base_s + np.random.uniform(0.5, 2.5)

                t_s_actual = (t_s_actual * 0.6) + (t_objetivo * 0.4)

                temp_s_futura.append(t_s_actual)
                temp_a_futura.append(t_s_actual + np.random.uniform(2.0, 4.0))

            np.random.seed(None)

            df_futuro = pd.DataFrame({
                'fecha': fechas_futuras,
                'humedad_superficial': hum_futura,
                'temp_suelo': temp_s_futura,
                'temp_aire': temp_a_futura,
                'precipitacion': lluvias_futuras
            })
            df = pd.concat([df_real_base, df_futuro], ignore_index=True)
        else:
            df = df_real_base.copy()

        dias_antes = df[(df['fecha'] >= inicio - pd.Timedelta(days=15)) & (df['fecha'] < inicio)]
        lluvia_15d_antes = dias_antes['precipitacion'].sum() if not dias_antes.empty else 0.0
        dias_despues = df[(df['fecha'] >= inicio) & (df['fecha'] <= inicio + pd.Timedelta(days=15))]

        t_actual = dias_despues['temp_suelo'].mean() if not dias_despues.empty else 0
        h_actual = dias_despues['humedad_superficial'].mean() if not dias_despues.empty else 0

        datos_dia_siembra = df[df['fecha'] == inicio]
        lluvia_hoy = datos_dia_siembra['precipitacion'].values[0] if not datos_dia_siembra.empty else 0.0

        if not en_fecha:
            reco_siembra = f"🛑 Fuera de época. El mes no pertenece a la ventana de siembra agronómica permitida."
            color_reco = "#dc3545"
        elif h_actual >= h_min and t_actual >= t_min:
            if lluvia_hoy > 3.0:
                reco_siembra = "✅ Ventana Habilitada. Condiciones óptimas, entrar a sembrar en cuanto el lote dé piso (riesgo de huella actual)."
                color_reco = "#28a745"
            else:
                reco_siembra, color_reco = ("✅ Ventana Habilitada. Lote en óptimas condiciones para sembrar en estos días.", "#28a745") if lluvia_15d_antes > 10 else ("✅ Ventana Habilitada. Humedad con lo justo pero garantiza germinación.", "#85c136")
        else:
            rango_busqueda = df[(df['fecha'] > inicio) & (df['fecha'] <= inicio + pd.Timedelta(days=10))]
            dia_optimo = None

            for index, row in rango_busqueda.iterrows():
                f_test = row['fecha']
                if f_test.month in meses_permitidos:
                    ventana_test = df[(df['fecha'] >= f_test) & (df['fecha'] <= f_test + pd.Timedelta(days=15))]
                    if not ventana_test.empty and ventana_test['humedad_superficial'].mean() >= h_min and ventana_test['temp_suelo'].mean() >= t_min:
                        dia_optimo = f_test
                        break

            if dia_optimo:
                dias_espera = (dia_optimo - inicio).days
                reco_siembra = f"⚠️ Alto riesgo hoy. Conviene esperar {dias_espera} días (aprox. {dia_optimo.strftime('%d/%m/%Y')}) a que mejoren las condiciones."
                color_reco = "#d39e00"
            else:
                if t_actual < t_min:
                    reco_siembra = f"🛑 Frío extremo. La temperatura del suelo no alcanzará los niveles mínimos requeridos."
                else:
                    reco_siembra = f"🛑 Riesgo Severo. No hay ventana óptima en los próximos 10 días. Evaluar cambio de estrategia."
                color_reco = "#dc3545"

        fecha_fin_calculo = inicio if inicio <= HOY else HOY
        fecha_inicio_calculo = fecha_fin_calculo - pd.Timedelta(days=30)
        datos_30_dias = df_real_base[(df_real_base['fecha'] >= fecha_inicio_calculo) & (df_real_base['fecha'] <= fecha_fin_calculo)]
        promedio_humedad_30d = datos_30_dias['humedad_superficial'].mean() if not datos_30_dias.empty else df_real_base['humedad_superficial'].mean()
        agua_util_perfil = np.clip((promedio_humedad_30d / 0.42) * 100, 10, 100)

        temp_suelo_display = datos_dia_siembra['temp_suelo'].values[0] if not datos_dia_siembra.empty else df['temp_suelo'].mean()

        try:
            idx_mes = meses_permitidos.index(mes_siembra)
            total_meses = len(meses_permitidos)
            es_tardio = idx_mes == (total_meses - 1)
            es_temprano = idx_mes == 0
        except ValueError:
            es_tardio = False
            es_temprano = False

        dias_proyeccion = df[(df['fecha'] >= inicio) & (df['fecha'] <= inicio + pd.Timedelta(days=45))]
        lluvia_proyectada = dias_proyeccion['precipitacion'].sum() if not dias_proyeccion.empty else 0.0
        temp_aire_proyectada = dias_proyeccion['temp_aire'].mean() if not dias_proyeccion.empty else 20.0

        if es_tardio:
            motivo = "Siembra en el límite de la ventana permitida. Hay alto riesgo de heladas tempranas al final del ciclo o estrés lumínico en el llenado de grano."
            reco_ciclo = f"Elegir <b>CICLO CORTO</b>. {motivo}"
            bg_ciclo, bord_ciclo, color_texto_ciclo = "#fff3cd", "#ffeeba", "#856404"
        elif agua_util_perfil < 45 and lluvia_proyectada < 40:
            motivo = f"Perfil desecado ({agua_util_perfil:.0f}%) y pronóstico futuro de bajas precipitaciones ({lluvia_proyectada:.0f} mm). El cultivo sufrirá estrés severo."
            reco_ciclo = f"Elegir <b>CICLO CORTO o HÍBRIDO DEFENSIVO</b>. {motivo} Conviene que la planta acelere sus etapas y demande menos agua total."
            bg_ciclo, bord_ciclo, color_texto_ciclo = "#f8d7da", "#f5c6cb", "#721c24"
        elif es_temprano and temp_aire_proyectada < 15:
            motivo = f"Siembra muy temprana con temperaturas futuras frías ({temp_aire_proyectada:.1f}°C promedio). Lento desarrollo vegetativo inicial."
            reco_ciclo = f"Elegir <b>CICLO LARGO</b>. {motivo} El cultivo necesita un ciclo extendido para compensar el arranque lento y maximizar kilos cuando el lote gane temperatura."
            bg_ciclo, bord_ciclo, color_texto_ciclo = "#cce5ff", "#b8daff", "#004085"
        elif agua_util_perfil > 65 and lluvia_proyectada >= 50:
            motivo = f"Excelente reserva hídrica actual ({agua_util_perfil:.0f}%) apoyada por buenas lluvias proyectadas ({lluvia_proyectada:.0f} mm)."
            reco_ciclo = f"Elegir <b>CICLO LARGO (o Intermedio-Largo)</b>. {motivo} Es el escenario ideal agronómicamente para buscar techos altos de rendimiento."
            bg_ciclo, bord_ciclo, color_texto_ciclo = "#d4edda", "#c3e6cb", "#155724"
        else:
            motivo = f"Condiciones agronómicas intermedias: Agua útil al {agua_util_perfil:.0f}%, lluvias futuras moderadas ({lluvia_proyectada:.0f} mm) y fecha de siembra óptima."
            reco_ciclo = f"Elegir <b>CICLO INTERMEDIO</b>. {motivo} Es la estrategia más equilibrada para balancear riesgo climático y potencial de rinde."
            bg_ciclo, bord_ciclo, color_texto_ciclo = "#e2e3e5", "#d6d8db", "#383d41"

        display(widgets.HTML(f"""
        <div style="display: flex; justify-content: space-between; background-color: #f8f9fa; padding: 15px; border-radius: 8px; border: 1px solid #dee2e6; margin-bottom: 10px;">
            <div style="text-align: center; border-right: 1px solid #ccc; padding-right: 15px; flex: 1;">
                <span style="font-size: 12px; color: #6c757d; text-transform: uppercase;">Temp. Suelo (-5cm)</span><br><strong style="font-size: 18px; color: #8c564b;">{temp_suelo_display:.1f} °C</strong>
            </div>
            <div style="text-align: center; border-right: 1px solid #ccc; padding: 0 15px; flex: 1;">
                <span style="font-size: 12px; color: #6c757d; text-transform: uppercase;">Lluvia Previa (-15 d)</span><br><strong style="font-size: 18px; color: #0d6efd;">{lluvia_15d_antes:.0f} mm</strong>
            </div>
            <div style="text-align: center; border-right: 1px solid #ccc; padding: 0 15px; flex: 1;">
                <span style="font-size: 12px; color: #6c757d; text-transform: uppercase;">Agua Útil (Perfil)</span><br><strong style="font-size: 18px; color: #1f77b4;">{agua_util_perfil:.0f} %</strong>
            </div>
            <div style="text-align: center; border-right: 1px solid #ccc; padding: 0 15px; flex: 1;">
                <span style="font-size: 12px; color: #6c757d; text-transform: uppercase;">Napa Freática</span><br><strong style="font-size: 18px; color: #17a2b8;">A {entrada_napa.value:.1f} m</strong>
            </div>
            <div style="text-align: center; padding-left: 15px; flex: 1.5;">
                <span style="font-size: 12px; color: #6c757d; text-transform: uppercase;">Decisión IA (Ventana +15d)</span><br><strong style="font-size: 14px; color: {color_reco};">{reco_siembra}</strong>
            </div>
        </div>

        <div style="background-color: {bg_ciclo}; padding: 12px 15px; border-radius: 8px; border: 1px solid {bord_ciclo}; margin-bottom: 15px; display: flex; align-items: center; gap: 12px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
            <span style="font-size: 24px;">🧬</span>
            <div>
                <span style="font-size: 12px; color: {color_texto_ciclo}; text-transform: uppercase;"><strong>Recomendación Genética Space Up</strong></span><br>
                <span style="font-size: 14px; color: #212529;">{reco_ciclo}</span>
            </div>
        </div>

        <h4 style='color: #0b192c; margin-top: 20px;'>🔍 Visor de Germinación (15 días antes y después de la siembra)</h4>
        """))

        df_micro = df[(df['fecha'] >= inicio - pd.Timedelta(days=15)) & (df['fecha'] <= inicio + pd.Timedelta(days=15))]
        fig1 = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1, row_heights=[0.5, 0.5], specs=[[{"secondary_y": True}], [{"secondary_y": False}]])
        fig1.add_trace(go.Bar(x=df_micro['fecha'], y=df_micro['precipitacion'], name='Lluvia (mm)', marker_color='#a0c4ff', opacity=0.7), row=1, col=1, secondary_y=True)
        fig1.add_trace(go.Scatter(x=df_micro['fecha'], y=df_micro['humedad_superficial'], name='Humedad Suelo', line=dict(color='#1f77b4', width=3)), row=1, col=1, secondary_y=False)
        fig1.add_hline(y=h_min, line_dash="dot", line_color="red", annotation_text=f"Mínimo Req. ({h_min})", row=1, col=1, secondary_y=False)
        fig1.add_trace(go.Scatter(x=df_micro['fecha'], y=df_micro['temp_suelo'], name='Temp. Suelo', line=dict(color='#8c564b', width=3)), row=2, col=1)
        fig1.add_hline(y=t_min, line_dash="dot", line_color="red", annotation_text=f"Mínimo Req. ({t_min}°C)", row=2, col=1)

        x_inicio_ms = inicio.timestamp() * 1000
        fig1.add_vline(x=x_inicio_ms, line_dash="solid", line_color="black", annotation_text="Día Siembra", row=1, col=1)
        fig1.add_vline(x=x_inicio_ms, line_dash="solid", line_color="black", row=2, col=1)

        fig1.update_layout(template="plotly_white", height=400, margin=dict(l=20, r=20, t=30, b=20), hovermode="x unified", showlegend=False)
        fig1.update_yaxes(title_text="Humedad", range=[0, 0.45], row=1, col=1, secondary_y=False)
        fig1.update_yaxes(title_text="Lluvia (mm)", range=[0, max(df_micro['precipitacion'].max() + 20, 50)], row=1, col=1, secondary_y=True, showgrid=False)
        fig1.update_yaxes(title_text="Temp (°C)", range=[0, 35], row=2, col=1)
        fig1.show(config={'displayModeBar': False})

def procesar_historico(b):
    global df, reporte_texto
    with out_historico:
        clear_output(wait=True)
        if df.empty or df['precipitacion'].sum() == 0:
            if marcador not in mapa.layers:
                display(widgets.HTML("<div style='color: #842029; background-color: #f8d7da; padding: 15px; border-radius: 8px;'>⚠️ <b>ALTO:</b> Ejecutá el Análisis de Siembra (Arriba) primero.</div>"))
                return
            descargar_api_con_respaldo(marcador.location[0], marcador.location[1], out_historico)
            df = df_real_base.copy()

        display(widgets.HTML("<div style='color: #052c65; background-color: #cfe2ff; padding: 15px; border-radius: 8px;'>📡 <i>Procesando histórico y evaluando estrés bioclimático...</i></div>"))
        clear_output(wait=True)

        inicio_hist = pd.to_datetime(fecha_siembra_hist.value)
        fin_hist = pd.to_datetime(fecha_cosecha_hist.value)

        if inicio_hist >= fin_hist:
            display(widgets.HTML("<div style='color: #842029; background-color: #f8d7da; padding: 15px; border-radius: 8px;'>⚠️ La fecha de cosecha debe ser posterior a la siembra.</div>"))
            return

        historial = [rot_1.value, rot_2.value, rot_3.value, rot_4.value, rot_5.value]

        # Identificar qué cultivo se está auditando según el año elegido en la UI
        y = selector_ano_hist.value
        if y == HOY.year - 5: cultivo_evaluado = rot_1.value
        elif y == HOY.year - 4: cultivo_evaluado = rot_2.value
        elif y == HOY.year - 3: cultivo_evaluado = rot_3.value
        elif y == HOY.year - 2: cultivo_evaluado = rot_4.value
        elif y == HOY.year - 1: cultivo_evaluado = rot_5.value
        else: cultivo_evaluado = 'Desconocido'

        cant_soja = historial.count('Soja')
        cant_gramineas = historial.count('Maíz') + historial.count('Trigo') + historial.count('Sorgo')
        cant_cobertura = historial.count('Pastura / Cultivo de Servicio')

        aporte_carbono = (cant_gramineas * 2.5) + (cant_cobertura * 3.0) + (cant_soja * 1.0)
        n_extraido = (cant_soja * 160) + (cant_gramineas * 110) + (cant_cobertura * 50)
        p_extraido = (cant_soja * 35) + (cant_gramineas * 25) + (cant_cobertura * 15)
        k_extraido = (cant_soja * 80) + (cant_gramineas * 90) + (cant_cobertura * 40)

        umbral = slider_umbral.value + (0.02 if 'Soja' in cultivo_evaluado else 0.0)

        ciclo = df[(df['fecha'] >= inicio_hist) & (df['fecha'] <= fin_hist)].copy()
        if ciclo.empty:
            display(widgets.HTML("<div style='color: #842029; background-color: #f8d7da; padding: 15px; border-radius: 8px;'>⚠️ Sin datos para estas fechas reales. Revisá el calendario.</div>"))
            return

        dias_tot = (fin_hist - inicio_hist).days
        dias_desde = (ciclo['fecha'] - pd.to_datetime(inicio_hist)).dt.days

        ndvi_ideal = 0.15 + 0.75 * np.exp(-0.5 * ((dias_desde - (dias_tot * 0.55)) / (dias_tot / 4.0))**2)

        estres_diario = np.where(ciclo['humedad_superficial'] < umbral, 0.015, 0.0)
        danio_acumulado = np.cumsum(estres_diario)

        ciclo['ndvi'] = np.clip(ndvi_ideal - danio_acumulado, 0.15, 0.95)

        df_macro = df.copy()
        df_macro['ndvi'] = 0.0
        df_macro.loc[df_macro['fecha'].isin(ciclo['fecha']), 'ndvi'] = ciclo['ndvi'].values

        dias_estres = len(ciclo[ciclo['humedad_superficial'] < umbral])
        porcentaje_estres = (dias_estres / len(ciclo)) * 100

        reporte_texto = f"""--- REPORTE TÉCNICO SPACE UP ---
Lote Analizado (Lat/Lon): {marcador.location[0]:.4f}, {marcador.location[1]:.4f}
Cultivo Auditado: {cultivo_evaluado}
Campaña Evaluada: {selector_ano_hist.label}
Periodo de Auditoría: {inicio_hist.strftime('%d/%m/%Y')} al {fin_hist.strftime('%d/%m/%Y')}
Secuencia Histórica de Rotación: {', '.join(historial)}

--- RESULTADOS BIOCLIMÁTICOS ---
Balance de Carbono Estimado: {aporte_carbono:.1f} t/ha equivalentes de rastrojo
Extracción Teórica N-P-K: N={n_extraido}kg, P={p_extraido}kg, K={k_extraido}kg
Días con Estrés Hídrico: {dias_estres} días ({porcentaje_estres:.1f}% del ciclo)
"""

        display(widgets.HTML(f"""
        <div style="font-family: sans-serif; padding: 20px; background-color: #f8f9fa; border: 1px solid #ced4da; border-radius: 8px; color: #212529; margin-bottom: 20px; box-shadow: 0 4px 10px rgba(0,0,0,0.03);">
            <h3 style="margin: 0 0 15px 0; font-size: 18px; color: #0b192c; border-bottom: 2px solid #0b192c; padding-bottom: 5px;">📊 Panel de Auditoría Histórica | {selector_ano_hist.label}</h3>
            <div style="display: flex; gap: 20px; font-size: 14px;">
                <div style="flex: 1;">
                    <p><strong>🌱 Cultivo Auditado:</strong> {cultivo_evaluado}</p>
                    <p><strong>📅 Periodo Evaluado:</strong> {inicio_hist.strftime('%d/%m/%Y')} al {fin_hist.strftime('%d/%m/%Y')}</p>
                    <p><strong>🔄 Secuencia Histórica:</strong> {', '.join(historial)}</p>
                    <p><strong>⚖️ Balance de Carbono (C):</strong> ~{aporte_carbono:.1f} t/ha equivalentes</p>
                </div>
                <div style="flex: 1;">
                    <p><strong>🧪 Extracción N-P-K:</strong> N: {n_extraido} kg/ha | P: {p_extraido} kg/ha | K: {k_extraido} kg/ha</p>
                    <p><strong>💧 Días bajo Umbral Hídrico:</strong> {dias_estres} días ({porcentaje_estres:.1f}% del ciclo)</p>
                    <p><strong>🛰️ Fuente de Datos:</strong> Telemetría Satelital & Archivo Histórico NASA POWER</p>
                </div>
            </div>
        </div>
        """))

        fecha_inic_g = pd.to_datetime(f"{max(2020, selector_ano_hist.value - 1)}-01-01")
        df_plot = df_macro[(df_macro['fecha'] >= fecha_inic_g) & (df_macro['fecha'] <= HOY)]

        fig2 = make_subplots(specs=[[{"secondary_y": True}]])
        fig2.add_trace(go.Bar(x=df_plot['fecha'], y=df_plot['precipitacion'], name='Lluvia Real (mm)', marker_color='rgba(54, 162, 235, 0.4)', hoverinfo='skip'), secondary_y=True)

        fig2.add_trace(go.Scatter(x=df_plot['fecha'], y=df_plot['humedad_superficial'], name='Humedad Previa', line=dict(color='#82c0cc', width=2, dash='dot')), secondary_y=False)
        fig2.add_trace(go.Scatter(x=ciclo['fecha'], y=ciclo['humedad_superficial'], name='Humedad en Ciclo', line=dict(color='#1f77b4', width=3)), secondary_y=False)
        fig2.add_trace(go.Scatter(x=ciclo['fecha'], y=ciclo['ndvi'], name=f'Desarrollo {cultivo_evaluado} (NDVI Bioclimático)', line=dict(color='#2ca02c', width=3), fill='tozeroy', fillcolor='rgba(44, 160, 44, 0.2)'), secondary_y=False)

        fig2.add_hline(y=umbral, line_dash="solid", line_color="#d62728", annotation_text="Límite Buffer", annotation_position="top left")

        x_in_ms, x_f_ms = inicio_hist.timestamp() * 1000, fin_hist.timestamp() * 1000
        fig2.add_vline(x=x_in_ms, line_dash="dash", line_color="#2ca02c", annotation_text="Siembra", annotation_position="bottom left")
        fig2.add_vline(x=x_f_ms, line_dash="dash", line_color="#8c564b", annotation_text="Cosecha", annotation_position="bottom right")

        fig2.update_layout(template="plotly_white", title=f"Telemetría Histórica - {selector_ano_hist.label} ({cultivo_evaluado})", height=550, hovermode="x unified", margin=dict(l=20, r=20, t=60, b=20), barmode='overlay', legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
        fig2.update_xaxes(range=[fecha_inic_g, fin_hist + pd.Timedelta(days=30)])
        fig2.update_yaxes(title_text="Humedad (0-0.5) / NDVI (0-1)", secondary_y=False, range=[0, 1.05])
        fig2.update_yaxes(title_text="Lluvia (mm)", secondary_y=True, range=[0, 150], showgrid=False)
        fig2.show(config={'scrollZoom': True, 'displayModeBar': True})

def exportar_datos(b):
    if reporte_texto != "":
        with open('Reporte_Auditoria_SpaceUp.txt', 'w', encoding='utf-8') as f:
            f.write(reporte_texto)
        files.download('Reporte_Auditoria_SpaceUp.txt')
    else:
        with out_historico: display(widgets.HTML("<div style='color: #842029; background-color: #f8d7da; padding: 15px; border-radius: 8px;'>⚠️ Ejecutá un análisis Histórico primero.</div>"))

boton_analizar_siembra.on_click(procesar_siembra)
boton_analizar_historico.on_click(procesar_historico)
boton_exportar.on_click(exportar_datos)

# ====================================================================
# 6. ESTRUCTURA DE LA INTERFAZ (DIVISIÓN EN 2 SECCIONES)
# ====================================================================
encabezado = widgets.HTML("<div style='background-color: #0b192c; padding: 20px; border-radius: 8px; color: white;'><h2>🛰️ Space Up Analytics</h2><p style='margin: 0; color: #a0b2c6;'>Sistema de Inteligencia Satelital Agronómica</p></div>")

# SECCIÓN 1 (ARRIBA - Siembra)
panel_mapa = widgets.VBox([widgets.HTML("<h4>📍 1. Ubicación Satelital</h4>"), caja_busqueda, salida_busqueda, mapa, widgets.HTML("<h4 style='margin-top:15px;'>🔍 Consulta Puntual (Datos Día)</h4>"), caja_consulta_puntual, salida_consulta_fecha], layout=widgets.Layout(width='45%', padding='20px', background_color='white', border_radius='10px'))
panel_parametros = widgets.VBox([widgets.HTML("<h4>⚙️ 2. Parámetros de Siembra Actual</h4>"), selector_cultivo, selector_fecha_siembra, entrada_napa, slider_umbral, widgets.HTML("<hr>"), boton_analizar_siembra], layout=widgets.Layout(width='45%', padding='20px', background_color='white', border_radius='10px'))
seccion_top = widgets.HBox([panel_mapa, panel_parametros], layout=widgets.Layout(justify_content='space-between'))

# SECCIÓN 2 (ABAJO - Rotación e Histórico)
panel_fechas_hist = widgets.VBox([widgets.HTML("<h4>📅 3.1. Campaña y Fechas Reales a Auditar</h4>"), selector_ano_hist, fecha_siembra_hist, fecha_cosecha_hist], layout=widgets.Layout(margin='0 0 15px 0'))
panel_rotacion = widgets.VBox([widgets.HTML("<h4>🔄 3.2. Historial de Rotación (Últimos 5 años previos)</h4>"), rot_1, rot_2, rot_3, rot_4, rot_5])
panel_hist_controles = widgets.VBox([panel_fechas_hist, panel_rotacion, widgets.HTML("<hr>"), widgets.HBox([boton_analizar_historico, boton_exportar])], layout=widgets.Layout(width='100%', padding='20px', background_color='white', border_radius='10px'))
seccion_bottom = widgets.VBox([widgets.HTML("<hr><h3 style='color: #0b192c; margin-top: 20px;'>📈 Sección 2: Auditoría Técnica Histórica</h3>"), panel_hist_controles, out_historico])

display(widgets.VBox([encabezado, seccion_top, out_siembra, seccion_bottom], layout=widgets.Layout(padding='25px', background_color='#f4f7f6')))

Inicializando Motor Analítico de Space Up...


In [ ]:
from google.colab import drive
drive.mount('/content/drive')